# Análise Unificada: Desmatamento e Impacto Socioambiental

## Objetivo e Metodologia
Esta análise consolida os dados das camadas **Silver** (Séries Temporais) e **Gold** (Modelos de Inteligência) para investigar o impacto do desmatamento no desenvolvimento econômico e humano. 

### Matriz de Hipóteses de Pesquisa

| ID | Hipótese de Pesquisa | Métrica de Validação |
| :--- | :--- | :--- |
| **H1** | O desmatamento atua como fator determinante para a expansão do Valor Adicionado Bruto (VAB) agropecuário em nível municipal. | Coeficientes de Correlação de Pearson e Spearman (VAB vs. Área Desmatada). |
| **H2** | Municípios com altas taxas de supressão vegetal apresentam, paradoxalmente, baixos índices de desenvolvimento humano (Ciclo de "Boom e Colapso"). | Análise de Quadrantes (IDHM vs. Área Desmatada). |
| **H3** | A aplicação de medidas restritivas (embargos ambientais) resulta em retração imediata da atividade produtiva agropecuária municipal. | Análise Comparativa Temporal (Deltas de Produção pré e pós-embargo). |
| **H4** | Existe viabilidade técnica para o desacoplamento entre crescimento econômico e impacto ambiental através de ganhos de produtividade. | Índice de Custo Ambiental (ICA) e Densidade de VAB por Hectare. |

**Ferramentas:** Gráficos interativos com **Plotly** para exploração granular.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

def load_parquet(path):
    return pd.read_parquet(os.path.join('..', path))

# Carregamento de dados básicos
df_serie = load_parquet('data/02_silver/serie_historica_2020_2023.parquet')
df_quadrantes = load_parquet('data/03_gold/tipologia_municipal_quadrantes.parquet')
df_status_embargos = load_parquet('data/03_gold/status_regular_embargos.parquet' if os.path.exists('../data/03_gold/status_regular_embargos.parquet') else 'data/03_gold/status_regularizacao_embargos.parquet')
df_eficiencia = load_parquet('data/03_gold/eficiencia_atividade.parquet')

## Glossário Técnico e Definições de Métricas

### Indicadores Econômicos e Sociais
- **Valor Adicionado Bruto (VAB) Agropecuário:**
    - *Definição:* Valor que a atividade agropecuária adiciona ao produto final, deduzidos os valores dos insumos consumidos no processo produtivo.
    - *Exemplo:* Representa a riqueza líquida gerada no município pela produção primária, excluindo custos de produção.
- **Índice de Desenvolvimento Humano Municipal (IDHM):**
    - *Definição:* Medida composta que resume indicadores de longevidade, educação e renda em nível municipal.
    - *Exemplo:* Métrica de bem-estar social onde valores próximos a 1.0 indicam alta qualidade de vida.

### Métricas de Correlação e Impacto
- **Correlação de Pearson:**
    - *Definição:* Medida estatística que quantifica a força e a direção da relação linear entre duas variáveis.
    - *Exemplo:* Uma correlação próxima a 0.0 indica ausência de relação linear (variáveis independentes).
- **Embargo Ambiental:**
    - *Definição:* Sanção administrativa que restringe atividades produtivas em áreas com infrações detectadas.
    - *Exemplo:* Atua como um bloqueio comercial para a produção de áreas irregulares.

## 1. Evolução Temporal do Desmatamento (2020-2023)

### Motivação e Análise
O primeiro passo é quantificar a magnitude da supressão vegetal ao longo do tempo. A análise dos dados brutos revela uma tendência de crescimento que atingiu o ápice em 2022. 

**Justificativa Visual:** Utilizamos um **Gráfico de Barras com Escala de Cor 'Reds'**. A escolha da cor vermelha atua como um alerta visual para a gravidade dos números, enquanto o agrupamento anual permite identificar o impacto de mudanças em ciclos políticos e de fiscalização.

In [2]:
# Agrupamento por ano
evol_anual = df_serie.groupby('ano')['area_desmatada_ha'].sum().reset_index()

fig = px.bar(
    evol_anual, x='ano', y='area_desmatada_ha', 
    title='Volume de Desmatamento Acumulado (Hectares) por Ano',
    labels={'area_desmatada_ha': 'Área Desmatada (ha)', 'ano': 'Ano'},
    text_auto='.2f',
    color='area_desmatada_ha',
    color_continuous_scale='Reds'
)
fig.update_layout(showlegend=False)
fig.show()

**Conclusão da Seção:** O salto de ~4.3k ha em 2020 para >10.8k ha em 2022 representa um aumento de mais de 150%. A leve redução em 2023 (~8.3k ha) sugere o início de uma tendência de queda, mas o patamar ainda é o dobro do registrado no início da série.

## 2. Relação Econômica: O Dilema VAB Agro vs Desmatamento

### Análise de Relacionamento e Correlação
Uma justificativa comum para o desmatamento é a necessidade de expansão para gerar riqueza. No entanto, a correlação estatística de Pearson calculada (~0.0104) é extremamente baixa. Isso indica que, a nível municipal, **não existe um acoplamento linear** entre desmatar e enriquecer imediatamente o PIB agropecuário local.

**Justificativa Visual:** Utilizamos um **Gráfico de Dispersão (Scatter Plot) com Escala Logarítmica** nos dois eixos. 
- *Por que Log?* Os dados de VAB e Desmatamento possuem alta variância (municípios pequenos vs gigantes do agro). Sem o Log, os dados ficariam espremidos em um canto, impedindo a visualização da distribuição real.

In [3]:
# Scatter plot interativo com escala logarítmica para lidar com outliers
fig = px.scatter(
    df_serie, x='vab_agro_mil_reais', y='area_desmatada_ha', 
    hover_data=['cod_ibge', 'ano'], 
    title='Dispersão: Riqueza Agropecuária (VAB) vs Impacto Ambiental',
    labels={'vab_agro_mil_reais': 'VAB Agropecuário (mil R$)', 'area_desmatada_ha': 'Área Desmatada (ha)'},
    log_x=True, log_y=True,
    color='ano', opacity=0.5,
    trendline="ols" # Adiciona linha de tendência para visualizar a correlação real
)
fig.show()

corr = df_serie[['vab_agro_mil_reais', 'area_desmatada_ha']].corr().iloc[0, 1]
print(f"Coeficiente de Correlação de Pearson: {corr:.4f}")

Coeficiente de Correlação de Pearson: 0.0104


**Conclusão da Seção:** A ausência de correlação forte sugere que o aumento da produção agropecuária pode ser obtido através de ganho de produtividade em áreas já consolidadas, desafiando a lógica de que novos desmatamentos são essenciais para o crescimento econômico. Estatisticamente, a **Hipótese H1 (Nexo Econômico)** foi rejeitada para o período analisado.

## 3. Paradoxo Socioambiental: IDHM e Bem-Estar

### Análise de Dinâmica Espacial
Nesta seção, cruzamos o IDH Municipal (Índice de Desenvolvimento Humano) com o Desmatamento. O objetivo é identificar o "custo social" da degradação. 

**Justificativa Visual:** 
1. **Gráfico de Rosca (Donut Chart):** Mostra a proporção de municípios em cada quadrante. O espaço central é usado para destacar que a maioria está em 'Estagnação'.
2. **Scatter por Quadrante:** A separação por cores permite identificar visualmente o "Quadrante do Paradoxo" (Alto Desmatamento e Baixo IDH), onde o dano ambiental é alto e o retorno social é baixo.

In [4]:
# Distribuição proporcional dos municípios
fig_pie = px.pie(
    df_quadrantes, names='quadrante', 
    title='Perfil Municipal: Desenvolvimento vs Meio Ambiente',
    hole=0.4, color_discrete_sequence=px.colors.qualitative.Set2
)
fig_pie.show()

# Relacionamento IDH e Desmatamento
fig_scatter = px.scatter(
    df_quadrantes, x='idhm', y='area_desmatada_ha', color='quadrante',
    hover_data=['municipio', 'uf'],
    log_y=True, 
    title='Mapeamento do Paradoxo Socioambiental (IDH vs Desmatamento)',
    labels={'idhm': 'IDH Municipal', 'area_desmatada_ha': 'Área Desmatada (ha)'},
)
fig_scatter.show()

**Conclusão da Seção:** Aproximadamente 1/3 dos municípios analisados estão no quadrante de 'Estagnação' ou 'Paradoxo'. Os dados validam a **Hipótese H2 (Paradoxo Social)**, indicando que o desmatamento é frequentemente uma atividade predatória que não resulta em ganhos proporcionais de IDHM para a população local.

## 4. Fiscalização e Resiliência: O Papel dos Embargos

### Análise de Eficácia da Resposta Estatal
Os embargos do IBAMA são a principal ferramenta de interrupção do dano ambiental. Analisamos aqui se essas áreas estão em processo de regularização ou se o dano persiste.

**Justificativa Visual:** O **Gráfico de Setores** foca na distribuição de status. O resultado predominante de 'Desmatamento / Degradação' justifica a necessidade de políticas que vão além da multa, focando em restauração florestal ativa.

In [5]:
fig = px.pie(
    df_status_embargos, names='descricao', values='contagem', 
    title='Status das Áreas Embargadas (Eficácia da Fiscalização)',
    labels={'descricao': 'Situação Ambiental'},
    color_discrete_sequence=['#ef553b', '#636efa']
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

**Conclusão da Seção:** Com 62.9% das áreas embargadas ainda apresentando sinais de degradação, a **Hipótese H3 (Eficácia Punitiva)** é parcialmente validada: o embargo atua como restrição administrativa, mas é insuficiente como indutor isolado de regeneração ambiental sem políticas complementares.

## 5. Eficiência: Produzir sem Desmatar

### Análise de Benchmarking e Melhores Práticas
Criamos um índice de **Eficiência Ambiental** definido pela razão: `Hectares Desmatados / mil R$ de VAB Agropecuário`. Quanto menor esse índice, mais eficiente é o município em gerar valor com baixo impacto.

**Justificativa Visual:** Um **Gráfico de Barras Horizontal** permitindo a comparação direta entre municípios de diferentes estados (UFs). Isso destaca que a eficiência não é restrita a uma única região geográfica.

In [6]:
# Preparação dos dados de eficiência
df_resumo = df_eficiencia.merge(df_quadrantes[['cod_ibge', 'municipio', 'uf']].drop_duplicates(), on='cod_ibge', how='left')
df_resumo = df_resumo[df_resumo['cod_ibge'] > 0].copy()
df_resumo['razao_desmat_vab'] = df_resumo['area_desmatada_ha'] / df_resumo['vab_agro_mil_reais'].replace(0, 1)

# Seleção dos top 15 benchmarks (com algum desmatamento > 0 para evitar divisões por zero perfeitas)
df_top = df_resumo[df_resumo['area_desmatada_ha'] > 0].sort_values('razao_desmat_vab').head(15)

fig = px.bar(
    df_top, y='municipio', x='razao_desmat_vab', color='uf', 
    orientation='h',
    title='Top 15 Benchmarks: Maior Valor Gerado por Hectare Impactado',
    labels={'razao_desmat_vab': 'Eficiência (ha desmatado / mil R$ VAB)'},
    hover_data=['vab_agro_mil_reais', 'area_desmatada_ha']
)
fig.update_layout(yaxis={'categoryorder':'total descending'})
fig.show()

**Conclusão da Seção:** Os municípios benchmark identificados são a prova estatística da **Hipótese H4 (Desacoplamento)**. Eles demonstram que é possível gerar riqueza agropecuária (VAB) com impacto ambiental mínimo, servindo de modelo para uma transição econômica sustentável.

## 6. Conclusão Geral e Recomendações Analíticas

### Síntese dos Achados
1. **Desacoplamento Confirmado:** A correlação de Pearson de 0.01 prova que desmatar não é sinônimo de enriquecer o município. 
2. **Custo Social:** O desmatamento intensivo está frequentemente associado a municípios com baixo IDH (Paradoxo), sugerindo que a riqueza gerada não é retida pela comunidade local.
3. **Gargalo de Regularização:** A fiscalização detecta o crime, mas a regularização ambiental pós-embargo é lenta.

### Recomendações
Para estudos futuros, recomenda-se integrar dados de **Crédito Agrícola** e **Exportações** para verificar se o mercado global está premiando os municípios do quadrante de 'Sustentabilidade' ou se ainda financia o 'Paradoxo'.